# SI7016 — Clase 03 — Lab A
## Agente con LangChain/LangGraph: tools + memoria

**Objetivo:** construir un agente con `create_agent`, inspeccionar tool calling y añadir memoria por checkpointing.

In [3]:
# secrets at google colab with userdata.get()
import os
from google.colab import userdata
if userdata.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("ANTHROPIC_API_KEY loaded from Colab Secrets and set in os.environ.")
elif userdata.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("OPENAI_API_KEY loaded from Colab Secrets and set in os.environ.")
else:
    print("Neither ANTHROPIC_API_KEY nor OPENAI_API_KEY found in Colab Secrets.")
    print("Please add one of them via the 🔑 Secrets panel on the left.")

In [4]:
import os
from dotenv import load_dotenv

# To fix the AssertionError, uncomment ONE of the lines below and replace 'YOUR_API_KEY_HERE' with your actual API key.
# os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY_HERE"
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

load_dotenv()

assert os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY"), "Please set either ANTHROPIC_API_KEY or OPENAI_API_KEY in your environment variables or .env file."
MODEL=os.getenv("HF_MODEL") or ("anthropic:claude-sonnet-4-6" if os.getenv("ANTHROPIC_API_KEY") else "openai:gpt-5.6")
print(MODEL)

## 1. Tools

In [5]:
from langchain.tools import tool
@tool
def buscar_curso(pregunta:str)->str:
    """Consulta información local de SI7016."""
    base={"temas":"NLP clásico, embeddings, Transformers, RAG y agentes.",
          "clase 03":"Frameworks, agentes, RAG, MCP/A2A, observabilidad y despliegue.",
          "proyecto":"Proyecto integrador de una solución NLP aplicada."}
    q=pregunta.lower()
    for k,v in base.items():
        if k in q: return v
    return "No encontré evidencia en la base local."
@tool
def contar_palabras(texto:str)->int:
    """Cuenta palabras."""
    return len(texto.split())

## 2. Agente

In [ ]:
!pip install langchain-anthropic

In [7]:
from langchain.agents import create_agent
agent=create_agent(model=MODEL,tools=[buscar_curso,contar_palabras],
    system_prompt="Eres un asistente de SI7016. Responde en español y usa tools cuando aporten evidencia o cálculo.")
r=agent.invoke({"messages":[{"role":"user","content":"¿Qué estudia la clase 03 y cuántas palabras tiene 'RAG conecta recuperación y generación'?"}]})
print(r["messages"][-1])

## 3. Inspeccionar la ejecución

In [8]:
for i,m in enumerate(r["messages"]):
    print("\n",i,m.type,m.content)
    if getattr(m,"tool_calls",None): print("tool_calls:",m.tool_calls)

## 4. Memoria con checkpointing

In [10]:
from langgraph.checkpoint.memory import InMemorySaver
agent_mem=create_agent(model=MODEL,tools=[buscar_curso])
cfg={"configurable":{"thread_id":"grupo-01"}}
agent_mem.invoke({"messages":[{"role":"user","content":"Mi tema de interés en esta conversación es RAG."}]},config=cfg)
r2=agent_mem.invoke({"messages":[{"role":"user","content":"¿Cuál era mi tema de interés?"}]},config=cfg)
print(r2["messages"][-1].content)

In [12]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "nvapi-xxx"
)

completion = client.chat.completions.create(
  model="openai/gpt-oss-120b",
  messages=[{"role":"user","content":"Which number is larger, 9.11 or 9.8?"}],
  temperature=1,
  top_p=1,
  max_tokens=4096,
  stream=False
)

reasoning = getattr(completion.choices[0].message, "reasoning_content", None)
if reasoning:
  print(reasoning)
print(completion.choices[0].message.content)

## Ejercicio
1. Agregue una tool real.

2. Compare dos `thread_id`.

3. Fuerce un caso sin tools.

4. Explique qué aporta LangGraph frente a LCEL lineal.

5. Extensión: cargue tools desde el Lab C.

### How to configure environment variables in Google Colab

There are two main ways to set environment variables in Google Colab:

1.  **For sensitive information (e.g., API Keys): Use Colab Secrets.**
    This is the most secure method, as it keeps your keys out of your notebook code.
    *   Click on the "🔑" (Secrets) icon in the left-hand panel.
    *   Click "Add new secret".
    *   Enter the `Name` of your environment variable (e.g., `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`).
    *   Enter the `Value` (your actual API key).
    *   Make sure "Notebook access" is enabled for your notebook.
    *   You can then access this secret in your Python code using `userdata.get('YOUR_SECRET_NAME')`.

2.  **For non-sensitive or temporary variables: Use `os.environ` in a code cell.**
    You can directly set environment variables within your notebook code using Python's `os` module.
    This is suitable for non-sensitive configurations or variables that change frequently.
    ```python
    import os
    os.environ['MY_VARIABLE'] = 'my_value'
    ```

For the `AssertionError` you encountered previously, you would use the first method (Colab Secrets) to set either `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`.

In [ ]:
# Import the userdata library to access Colab secrets
from google.colab import userdata

# Example of how to load an API key from Colab Secrets
# Replace 'YOUR_API_KEY_NAME' with the actual name you used in the Secrets panel
# For example, if you named it 'ANTHROPIC_API_KEY', you would use 'ANTHROPIC_API_KEY' below.

# You would uncomment one of these lines and replace the placeholder.
# anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
# openai_api_key = userdata.get('OPENAI_API_KEY')

# To demonstrate, let's assume you set 'ANTHROPIC_API_KEY' in your secrets.
# We'll then set it in os.environ, which is what the original code expects.
if userdata.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("ANTHROPIC_API_KEY loaded from Colab Secrets and set in os.environ.")
elif userdata.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("OPENAI_API_KEY loaded from Colab Secrets and set in os.environ.")
else:
    print("Neither ANTHROPIC_API_KEY nor OPENAI_API_KEY found in Colab Secrets.")
    print("Please add one of them via the 🔑 Secrets panel on the left.")

# You can verify if it's set (only if you've added it to secrets):
# print(os.getenv("ANTHROPIC_API_KEY"))
# print(os.getenv("OPENAI_API_KEY"))

# After running this, the original cell (T3uOlkg8K9_R) should execute without the AssertionError,
# assuming you have provided a valid API key in the Colab Secrets manager.